# nb01 - Clean: CMS Star Ratings

**Run locally after nb00.** All raw-to-clean transformation for this project lives in this notebook, for reproducibility.

Two outputs:
1. `../data/star_summary_clean.csv` - one row per contract per year (Overall / Part C / Part D stars)
2. `../data/star_measures_clean.csv` - one row per contract per year **per measure**, with its Domain

**File quirks handled here:**
- The CMS CSVs are Windows-1252 encoded, not UTF-8 (smart apostrophes in measure names).
- Every file has a **title row** on top.
- The Measure Stars file has a **two-row header**: the upper row holds the Domain names (merged across columns, so they read as blanks), and the row beneath it holds the actual measure names.

In [1]:
import json
from pathlib import Path
import pandas as pd

DATA = Path('..') / 'data'
RAW = DATA / 'raw'
years = [f['year'] for f in json.loads((DATA / 'extraction_log.json').read_text())['files']]
print('years:', years)

def find_csv(yr, needle):
    hits = [p for p in (RAW / yr).rglob('*.csv') if needle in p.name]
    assert len(hits) == 1, (yr, needle, [h.name for h in hits])
    return hits[0]

def read_cms_csv(path, **kw):
    # CMS files are Windows-1252, not UTF-8
    for enc in ('utf-8', 'cp1252', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f'could not decode {path}')

def star_to_num(series):
    # leading star number (1 to 5, half steps); text like 'Not enough data available' -> NaN
    return pd.to_numeric(series.astype(str).str.extract(r'(\d(?:\.\d)?)')[0], errors='coerce')

years: ['2024', '2025', '2026']


## Part 1 - Summary ratings (one row per contract per year)

In [2]:
frames = []
for yr in years:
    f = find_csv(yr, 'Summary Rating')
    df = read_cms_csv(f, skiprows=1, dtype=str)   # skip the title row
    df.columns = [str(c).strip() for c in df.columns]

    partc   = [c for c in df.columns if 'Part C Summary' in c][0]
    partd   = [c for c in df.columns if 'Part D Summary' in c][0]
    overall = [c for c in df.columns if c.strip().endswith('Overall')][0]

    out = pd.DataFrame({
        'Year': int(yr),
        'Contract':      df['Contract Number'].str.strip(),
        'OrgType':       df['Organization Type'].str.strip(),
        'ContractName':  df['Contract Name'].str.strip(),
        'MarketingName': df['Organization Marketing Name'].str.strip(),
        'ParentOrg':     df['Parent Organization'].str.strip(),
        'SNP':           df['SNP'].str.strip() if 'SNP' in df.columns else pd.NA,
        'PartC_Stars':   star_to_num(df[partc]),
        'PartD_Stars':   star_to_num(df[partd]),
        'Overall_Stars': star_to_num(df[overall]),
    })
    print(f'{yr}: {len(out)} rows | overall rated: {out.Overall_Stars.notna().sum()}')
    frames.append(out)

stars = pd.concat(frames, ignore_index=True)
stars.to_csv(DATA / 'star_summary_clean.csv', index=False)
print('\nwrote star_summary_clean.csv', stars.shape)
print('\nnational average OVERALL star per year (the benchmark):')
print(stars.groupby('Year').Overall_Stars.mean().round(3))

2024: 857 rows | overall rated: 545
2025: 789 rows | overall rated: 521
2026: 769 rows | overall rated: 516

wrote star_summary_clean.csv (2415, 10)

national average OVERALL star per year (the benchmark):
Year
2024    3.682
2025    3.650
2026    3.652
Name: Overall_Stars, dtype: float64


## Part 2 - Measure-level stars (two-row header handled)

Read with no header, then rebuild the column names: the ID columns take their names from the upper (Domain) row, and every measure column takes its name from the lower (Measure) row. The Domain row is forward filled so each measure keeps the domain it belongs to. Then unpivot wide to long.

In [3]:
mframes = []
for yr in years:
    f = find_csv(yr, 'Measure Stars')
    raw = read_cms_csv(f, skiprows=1, header=None, dtype=str)   # skip title; keep both header rows as data

    hdr_domain  = raw.iloc[0]   # upper header row: ID names + Domain names (merged -> blanks)
    hdr_measure = raw.iloc[1]   # lower header row: blanks over the IDs, then measure names

    # the leading columns where the MEASURE row is blank are the ID columns
    def _blank(v):
        return pd.isna(v) or str(v).strip() == ''
    n_id = 0
    while n_id < len(hdr_measure) and _blank(hdr_measure.iloc[n_id]):
        n_id += 1
    if n_id == 0:
        n_id = 5   # fallback

    id_names      = [str(hdr_domain.iloc[i]).strip() for i in range(n_id)]
    measure_names = [str(hdr_measure.iloc[i]).strip() for i in range(n_id, raw.shape[1])]
    # each measure column inherits the domain above it (forward fill across the merged cells)
    domains = (hdr_domain.iloc[n_id:].ffill().fillna('').astype(str).str.strip().tolist())
    measure_to_domain = dict(zip(measure_names, domains))

    data = raw.iloc[2:].reset_index(drop=True)
    data.columns = id_names + measure_names

    # strip whitespace on the ID values (contract IDs carried trailing spaces)
    for c in id_names:
        data[c] = data[c].astype(str).str.strip().replace({'nan': pd.NA, '': pd.NA})

    print(f'\n{yr}: {n_id} ID columns, {len(measure_names)} measure columns')
    for m in measure_names[:5]:
        print(f'     {m}   [domain: {measure_to_domain[m][:40]}]')
    print(f'     ... and {len(measure_names)-5} more')

    id_col = 'CONTRACT_ID' if 'CONTRACT_ID' in id_names else 'Contract Number'
    long = data.melt(id_vars=id_names, value_vars=measure_names,
                     var_name='MeasureLabel', value_name='StarsRaw')
    long = long.rename(columns={id_col: 'Contract',
                                'Organization Type': 'OrgType',
                                'Organization Marketing Name': 'MarketingName',
                                'Parent Organization': 'ParentOrg'})
    long['Year'] = int(yr)
    long['Domain'] = long['MeasureLabel'].map(measure_to_domain)

    sp = long['MeasureLabel'].str.split(':', n=1, expand=True)
    long['MeasureCode'] = sp[0].str.strip()
    long['MeasureName'] = sp[1].str.strip() if sp.shape[1] > 1 else long['MeasureLabel']
    long['Stars'] = star_to_num(long['StarsRaw'])

    keep = ['Year','Contract','OrgType','MarketingName','ParentOrg',
            'Domain','MeasureCode','MeasureName','StarsRaw','Stars']
    mframes.append(long[[c for c in keep if c in long.columns]])

measures = pd.concat(mframes, ignore_index=True)
measures.to_csv(DATA / 'star_measures_clean.csv', index=False)
print('\nwrote star_measures_clean.csv', measures.shape)
print('rated measure rows:', measures.Stars.notna().sum())
print('sample contracts:', list(measures.Contract.dropna().unique()[:5]))
print('H1224 rows found:', int((measures.Contract == 'H1224').sum()))


2024: 5 ID columns, 46 measure columns
     C01: Breast Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C02: Colorectal Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C03: Annual Flu Vaccine   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C04: Monitoring Physical Activity   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C05: Special Needs Plan (SNP) Care Management   [domain: HD2: Managing Chronic (Long Term) Condit]
     ... and 41 more

2025: 5 ID columns, 46 measure columns
     C01: Breast Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C02: Colorectal Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C03: Annual Flu Vaccine   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C04: Monitoring Physical Activity   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C05: Special Needs Plan (SNP) Care Management   [domain: HD2: Managing Chronic (Long T

## QA - where does L.A. Care (H1224) lose stars?

In [4]:
LA = 'H1224'

print('=== L.A. Care overall stars by year ===')
print(stars[stars.Contract == LA][['Year','MarketingName','Overall_Stars','PartC_Stars','PartD_Stars']].to_string(index=False))

yr_latest = max(int(y) for y in years)
la_rows = measures[(measures.Contract == LA) & (measures.Year == yr_latest)]
print(f'\nL.A. Care rows in {yr_latest}: {len(la_rows)} | rated: {la_rows.Stars.notna().sum()}')
print('sample raw values:', list(la_rows.StarsRaw.dropna().unique()[:6]))

nat = (measures[measures.Stars.notna()]
       .groupby(['Year','MeasureCode','MeasureName'])['Stars'].mean().round(2).rename('NationalAvg'))
la  = (la_rows.set_index(['Year','MeasureCode','MeasureName'])['Stars'].rename('LACare'))

cmp = pd.concat([la, nat], axis=1, join='inner').dropna(subset=['LACare'])
cmp['Gap'] = (cmp['LACare'] - cmp['NationalAvg']).round(2)

print(f'\n=== {yr_latest}: measures where L.A. Care is FURTHEST BELOW the national average ===')
print(cmp.sort_values('Gap').head(12).to_string())

print(f'\n=== {yr_latest}: measures where L.A. Care is ABOVE the national average ===')
print(cmp.sort_values('Gap', ascending=False).head(8).to_string())

print('\n=== L.A. Care average stars by domain ===')
print(la_rows[la_rows.Stars.notna()].groupby('Domain').Stars.agg(['mean','count']).round(2).to_string())

=== L.A. Care overall stars by year ===
 Year         MarketingName  Overall_Stars  PartC_Stars  PartD_Stars
 2024 L.A. Care Health Plan            NaN          NaN          NaN
 2025 L.A. Care Health Plan            3.0          2.5          3.5
 2026 L.A. Care Health Plan            3.0          3.0          4.0

L.A. Care rows in 2026: 45 | rated: 43
sample raw values: ['2', '3', '4', 'Plan too new to be measured ', '5', '1']

=== 2026: measures where L.A. Care is FURTHEST BELOW the national average ===
                                                            LACare  NationalAvg   Gap
Year MeasureCode MeasureName                                                         
2026 C17         Medication Reconciliation Post-Discharge      1.0         3.82 -2.82
     C27         Care Coordination                             1.0         3.49 -2.49
     C24         Customer Service                              1.0         3.46 -2.46
     C08         Care for Older Adults – Medication Review